In [1]:
import arcpy
import os
arcpy.env.overwriteOutput = True
arcpy.env.outputCoordinateSystem = None
project_folder = r"C:/Users/Chels/OneDrive - University of Illinois - Urbana/Dissertation/Ch4_Sackett_Impacts/Wetland Mapping"
wetland_gdb = os.path.normpath(os.path.join(project_folder, "Illinois_Wetland_Shapefiles.gdb"))
waters_gdb = os.path.normpath(os.path.join(project_folder, "Jurisdictional_Water_Shapefiles.gdb"))

## Intersect NWI with NLD leveed areas layer

In [2]:
# NLCD years
years = [2023, 2024]
n_y = len(years)

In [3]:
# path to NLD leveed areas layer
leveed_areas = os.path.normpath(os.path.join(project_folder, r"Wetland Mapping.gdb/LeveedAreas_Clip_AlbersNAD1983"))
levee_field = "Within_Lev"

In [4]:
# all wetlands above area threshold
step11 = [os.path.normpath(os.path.join(wetland_gdb, f"IL_WS_Step11_{y}NLCD_AreaThreshold")) for y in years]

In [5]:
for i in range(n_y):
    # create levee intersect field
    if not arcpy.ListFields(step11[i], levee_field):
        arcpy.management.AddField(step11[i], levee_field, "LONG");

    # pre-fill field with zeros
    arcpy.management.CalculateField(
        in_table=step11[i],
        field=levee_field,
        expression="0",
        expression_type="PYTHON3"
    );

    # select all wetland polygons that occur within leveed area
    arcpy.management.MakeFeatureLayer(step11[i], "temp_layer");
    arcpy.management.SelectLayerByLocation(
        in_layer="temp_layer",
        overlap_type="WITHIN",
        select_features=leveed_areas,
        selection_type="NEW_SELECTION"
    );
    count = int(arcpy.management.GetCount("temp_layer")[0])
    print(f"Selected: {count} features")

    # calculate 1 only for selected features
    arcpy.management.CalculateField(
        in_table="temp_layer",
        field=levee_field,
        expression="1",
        expression_type="PYTHON3"
    );
    arcpy.management.Delete("temp_layer");

Selected: 7469 features
Selected: 7506 features


## Intersect NWI with jurisdictional waters layers

In [8]:
# buffer distances
buffers = [1, 5, 10, 20, 100]
n_b = len(buffers)

In [9]:
# flow permanence levels
permanence_levels = [1, 2, 3]
n_p = len(permanence_levels)

In [25]:
# create old NHD fields for each flow permanence and buffer distance combination
nhd_fields = [f"NHDIn{l}_{b}" for l in permanence_levels for b in buffers] + [f"NHDIn{l}_20V" for l in permanence_levels]
nhd_layers = [f"Jurisdictional_Waters_{l}_{b}" for l in permanence_levels for b in buffers] + [f"Jurisdictional_Waters_{l}_20_Variable" for l in permanence_levels]
nhd_paths = [os.path.normpath(os.path.join(waters_gdb, f"{layer}")) for layer in nhd_layers]

In [26]:
nhd_fields

['NHDIn1_1',
 'NHDIn1_5',
 'NHDIn1_10',
 'NHDIn1_20',
 'NHDIn1_100',
 'NHDIn2_1',
 'NHDIn2_5',
 'NHDIn2_10',
 'NHDIn2_20',
 'NHDIn2_100',
 'NHDIn3_1',
 'NHDIn3_5',
 'NHDIn3_10',
 'NHDIn3_20',
 'NHDIn3_100',
 'NHDIn1_20V',
 'NHDIn2_20V',
 'NHDIn3_20V']

In [27]:
nhd_layers

['Jurisdictional_Waters_1_1',
 'Jurisdictional_Waters_1_5',
 'Jurisdictional_Waters_1_10',
 'Jurisdictional_Waters_1_20',
 'Jurisdictional_Waters_1_100',
 'Jurisdictional_Waters_2_1',
 'Jurisdictional_Waters_2_5',
 'Jurisdictional_Waters_2_10',
 'Jurisdictional_Waters_2_20',
 'Jurisdictional_Waters_2_100',
 'Jurisdictional_Waters_3_1',
 'Jurisdictional_Waters_3_5',
 'Jurisdictional_Waters_3_10',
 'Jurisdictional_Waters_3_20',
 'Jurisdictional_Waters_3_100',
 'Jurisdictional_Waters_1_20_Variable',
 'Jurisdictional_Waters_2_20_Variable',
 'Jurisdictional_Waters_3_20_Variable']

In [28]:
# create old Brinkerhoff fields for each flow permanence and buffer distance combination
brink_fields = [f"BRFIn{l}_{b}" for l in permanence_levels for b in buffers] + [f"BRFIn{l}_20V" for l in permanence_levels]
brink_layers = [f"Jurisdictional_Waters_Brinkerhoff_{l}_{b}" for l in permanence_levels for b in buffers] + [f"Jurisdictional_Waters_Brinkerhoff_{l}_20_Variable" for l in permanence_levels]
brink_paths = [os.path.normpath(os.path.join(waters_gdb, f"{layer}")) for layer in brink_layers]

In [29]:
brink_fields

['BRFIn1_1',
 'BRFIn1_5',
 'BRFIn1_10',
 'BRFIn1_20',
 'BRFIn1_100',
 'BRFIn2_1',
 'BRFIn2_5',
 'BRFIn2_10',
 'BRFIn2_20',
 'BRFIn2_100',
 'BRFIn3_1',
 'BRFIn3_5',
 'BRFIn3_10',
 'BRFIn3_20',
 'BRFIn3_100',
 'BRFIn1_20V',
 'BRFIn2_20V',
 'BRFIn3_20V']

In [30]:
brink_layers

['Jurisdictional_Waters_Brinkerhoff_1_1',
 'Jurisdictional_Waters_Brinkerhoff_1_5',
 'Jurisdictional_Waters_Brinkerhoff_1_10',
 'Jurisdictional_Waters_Brinkerhoff_1_20',
 'Jurisdictional_Waters_Brinkerhoff_1_100',
 'Jurisdictional_Waters_Brinkerhoff_2_1',
 'Jurisdictional_Waters_Brinkerhoff_2_5',
 'Jurisdictional_Waters_Brinkerhoff_2_10',
 'Jurisdictional_Waters_Brinkerhoff_2_20',
 'Jurisdictional_Waters_Brinkerhoff_2_100',
 'Jurisdictional_Waters_Brinkerhoff_3_1',
 'Jurisdictional_Waters_Brinkerhoff_3_5',
 'Jurisdictional_Waters_Brinkerhoff_3_10',
 'Jurisdictional_Waters_Brinkerhoff_3_20',
 'Jurisdictional_Waters_Brinkerhoff_3_100',
 'Jurisdictional_Waters_Brinkerhoff_1_20_Variable',
 'Jurisdictional_Waters_Brinkerhoff_2_20_Variable',
 'Jurisdictional_Waters_Brinkerhoff_3_20_Variable']

In [31]:
# NHD: create new fields and fill with zeros
for i in range(n_y):
    for f in nhd_fields:
        existing = [f.name for f in arcpy.ListFields(step11[i])]
        if f not in existing:
            arcpy.management.AddField(step11[i], f, "LONG");            
            arcpy.management.CalculateField(
                in_table=step11[i],
                field=f,
                expression="0",
                expression_type="PYTHON3"
            );

In [35]:
# Brinkerhoff: create new fields and fill with zeros
for i in range(n_y):
    for f in brink_fields:
        existing = [f.name for f in arcpy.ListFields(step11[i])]
        if f not in existing:
            arcpy.management.AddField(step11[i], f, "LONG");      
            arcpy.management.CalculateField(
                in_table=step11[i],
                field=f,
                expression="0",
                expression_type="PYTHON3"
            );

In [38]:
# NHD: set field values to 1 if they intersect with jurisdictional waters
for i in range(n_y):
    for j in range(len(nhd_fields)):
        arcpy.management.MakeFeatureLayer(step11[i], "temp_layer");
        arcpy.management.SelectLayerByLocation(
            in_layer="temp_layer",
            overlap_type="INTERSECT",
            select_features=nhd_paths[j],
            selection_type="NEW_SELECTION"
        );
        arcpy.management.CalculateField(
            in_table="temp_layer", 
            field=nhd_fields[j],
            expression="1",
            expression_type="PYTHON3"
        );
        arcpy.management.Delete("temp_layer");

In [39]:
for i in range(n_y):
    for j in range(len(brink_fields)):
        arcpy.management.MakeFeatureLayer(step11[i], "temp_layer");
        arcpy.management.SelectLayerByLocation(
            in_layer="temp_layer",
            overlap_type="INTERSECT",
            select_features=brink_paths[j],
            selection_type="NEW_SELECTION"
        );
        arcpy.management.CalculateField(
            in_table="temp_layer",      # calculates on selection only
            field=brink_fields[j],
            expression="1",
            expression_type="PYTHON3"
        );
        arcpy.management.Delete("temp_layer");